# 第 6 周练习 —— 客服工单优先级分类器（Fine-tuning 流水线）

## 练习目标（理念）

构建一条 **微调（fine-tuning）准备流水线**：把客服工单自动分成 **Urgent / High / Medium / Low** 四档优先级。

- **场景**：帮助支持团队快速分流进线工单
- **本练习重点**：合成数据 → 划分 → **零样本基线（zero-shot baseline）** → 导出 JSONL → Gradio 交互评估
- **对照**：先量出 frontier 模型不微调时的准确率，再谈「微调要超过这个数」

## 和本课第 6 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 数据整理与预处理 | `TICKETS` 合成集 + train/val/test 划分 |
| 评估指标与基线 | `predict_baseline` + `calculate_accuracy` |
| Frontier 模型调用 | OpenRouter / OpenAI `chat.completions.create` |
| 微调数据格式 | JSONL：`messages` = system / user / assistant |
| Gradio UI | 单条分类、批量评估、导出状态 |

## 怎么跑

1. 准备 `.env`：`OPENROUTER_API_KEY`（优先）或 `OPENAI_API_KEY`
2. 从上到下运行单元格；先看基线准确率
3. 需要时导出 `jsonl/`，再用 OpenAI / 开源工具真正微调


In [1]:
# ========== 导入：环境、JSON、随机划分、OpenAI、Gradio ==========

# 标准库 os：读环境变量、建目录、拼路径
import os
# 标准库 json：序列化 messages → JSONL 行
import json
# 标准库 random：打乱数据集（配合 seed 可复现）
import random
# 标准库 re：清洗模型偶尔包上的 ```json 代码块
import re
# Counter：统计各优先级出现次数
from collections import Counter
# load_dotenv：把 .env 读进进程环境，避免把密钥写进笔记本
from dotenv import load_dotenv
# OpenAI 客户端：也可通过 base_url 指向 OpenRouter 兼容接口
from openai import OpenAI
# Gradio：搭交互式分类 / 评估界面
import gradio as gr

# override=True：.env 里的值覆盖已有同名环境变量
load_dotenv(override=True)


True

In [2]:
# ========== 配置：优先 OpenRouter，否则直连 OpenAI ==========

# OpenRouter 的 OpenAI 兼容 API 根地址
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
# OpenRouter 上的模型路由名（带厂商前缀 openai/）
FRONTIER_MODEL = "openai/gpt-4.1-mini"

# 先读 OPENROUTER_API_KEY；有则走 OpenRouter
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    # base_url 指向 OpenRouter；api_key 用 OpenRouter 的 key
    client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)
    print("Using OpenRouter API")
else:
    # 回退：默认 OpenAI 云端；模型名改为不带 openai/ 前缀的官方 id
    client = OpenAI()
    FRONTIER_MODEL = "gpt-4.1-mini"
    print("Using OpenAI API directly")


Using OpenRouter API


## 数据集：客服工单（合成）

每条样本是「工单英文正文 + 优先级标签」。标签含义：

| 优先级 | 含义 |
|--------|------|
| **Urgent** | 系统宕机、安全漏洞、数据丢失、生产中断 |
| **High** | 核心功能坏了、支付异常、大量用户受影响 |
| **Medium** | 功能需求、小 bug、性能担忧 |
| **Low** | 一般咨询、文档问题、外观细节 |


In [3]:
# ========== 合成客服工单数据集：四档优先级标签 ==========
# PRIORITIES：合法标签集合（评估时用 in / 循环匹配）
PRIORITIES = ("Urgent", "High", "Medium", "Low")

# TICKETS：合成训练集；ticket / priority 字符串保持英文原样（影响训练与评估）
TICKETS = [
    # ----- Urgent：宕机 / 安全 / 数据丢失 -----
    {"ticket": "CRITICAL: Our entire production database is down. No users can access the application. This is affecting 50,000+ customers right now. Need immediate assistance.", "priority": "Urgent"},
    {"ticket": "SECURITY ALERT: We detected unauthorized access to our admin panel. Someone from an unknown IP accessed sensitive customer data. Need to lock down immediately.", "priority": "Urgent"},
    {"ticket": "Data corruption detected in customer payment records. Transactions from the last 24 hours may be affected. This is a compliance issue.", "priority": "Urgent"},
    {"ticket": "Complete service outage. API returning 500 errors for all endpoints. Our mobile app is completely unusable. Revenue impact: $10K/hour.", "priority": "Urgent"},
    {"ticket": "RANSOMWARE ALERT: Our servers have been encrypted. Attackers demanding payment. All systems locked. Business completely halted.", "priority": "Urgent"},
    {"ticket": "Production server crashed and won't restart. Auto-recovery failed. All customer-facing services are offline. SLA breach imminent.", "priority": "Urgent"},
    {"ticket": "Critical vulnerability discovered in authentication system. Attackers can bypass login. Patching required immediately before exploitation.", "priority": "Urgent"},
    {"ticket": "Payment processing completely broken. All transactions failing. Customers cannot complete purchases. E-commerce site effectively down.", "priority": "Urgent"},
    {"ticket": "DDoS attack in progress. Our infrastructure is being overwhelmed. Website unreachable for last 30 minutes. Need mitigation NOW.", "priority": "Urgent"},
    {"ticket": "Backup system failed during restore. Lost 48 hours of customer data. Critical business data unrecoverable without help.", "priority": "Urgent"},
    {"ticket": "SSL certificate expired on production. Chrome showing security warning to all users. Customers afraid to enter payment info.", "priority": "Urgent"},
    {"ticket": "Database replication failed. Primary and secondary out of sync. Risk of data loss if primary fails. Need immediate intervention.", "priority": "Urgent"},
    
    # ----- High：核心功能坏 / 支付 / 大面积影响 -----
    {"ticket": "User login is failing for about 30% of our customers. They're getting 'invalid credentials' even with correct passwords. Started 2 hours ago.", "priority": "High"},
    {"ticket": "Credit card payments are being declined randomly. Some customers charged twice. Others not charged but order confirmed. Very confusing.", "priority": "High"},
    {"ticket": "Email notifications stopped working completely. Customers not receiving order confirmations, password resets, or any transactional emails.", "priority": "High"},
    {"ticket": "Search functionality broken on our e-commerce site. Returns no results for any query. Customers can't find products.", "priority": "High"},
    {"ticket": "Mobile app crashes immediately on launch for iOS 17 users. About 40% of our user base affected. App store reviews tanking.", "priority": "High"},
    {"ticket": "Checkout process broken. Users can add items to cart but 'Place Order' button does nothing. No error message shown.", "priority": "High"},
    {"ticket": "User dashboard showing wrong data. Customers seeing other users' information. Privacy concern and potential GDPR issue.", "priority": "High"},
    {"ticket": "API rate limiting is too aggressive. Legitimate integrations being blocked. Partners complaining they can't sync data.", "priority": "High"},
    {"ticket": "File upload feature broken. Users trying to upload documents get timeout errors. Critical for our document management workflow.", "priority": "High"},
    {"ticket": "Subscription renewal failing for annual plans. Customers being downgraded to free tier unexpectedly. Billing team overwhelmed.", "priority": "High"},
    {"ticket": "Two-factor authentication not sending codes. Users locked out of accounts. Password reset also requires 2FA. Catch-22 situation.", "priority": "High"},
    {"ticket": "Report generation timing out for large datasets. Enterprise customers can't generate monthly reports. Contract SLA at risk.", "priority": "High"},
    {"ticket": "Webhook deliveries failing. Integration partners not receiving events. Their automations broken due to our issue.", "priority": "High"},
    {"ticket": "User permissions not applying correctly. Some users have admin access who shouldn't. Security review needed urgently.", "priority": "High"},
    
    # ----- Medium：需求 / 小 bug / 性能 -----
    {"ticket": "Page load time increased from 2s to 5s after last update. Not critical but users noticing slowdown. Would like investigation.", "priority": "Medium"},
    {"ticket": "Feature request: Can we add dark mode to the dashboard? Many users asking for this. Would improve user experience.", "priority": "Medium"},
    {"ticket": "Export to PDF not formatting tables correctly. Data is there but alignment is off. Low priority but would be nice to fix.", "priority": "Medium"},
    {"ticket": "Calendar integration with Google Calendar showing events 1 hour off. Timezone issue probably. Can work around by adjusting.", "priority": "Medium"},
    {"ticket": "Search results could be more relevant. Looking for ways to improve our search algorithm. Not broken, just could be better.", "priority": "Medium"},
    {"ticket": "Mobile app battery usage seems high. Users reporting drain. Would like optimization but app still functional.", "priority": "Medium"},
    {"ticket": "Feature request: Add bulk edit capability for user management. Currently have to edit one by one. Would save admin time.", "priority": "Medium"},
    {"ticket": "Charts on analytics dashboard not responsive on tablet. Data visible but layout breaks. Desktop and mobile work fine.", "priority": "Medium"},
    {"ticket": "Notification sounds not working on Android. Visual notifications work. Minor issue but some users prefer audio alerts.", "priority": "Medium"},
    {"ticket": "Auto-save feature sometimes takes 10+ seconds. Usually instant. Intermittent issue, can't reproduce consistently.", "priority": "Medium"},
    {"ticket": "Feature request: Would like to customize email templates. Currently using default designs. Branding consistency important.", "priority": "Medium"},
    {"ticket": "Memory usage gradually increases over time. Have to refresh browser after 4-5 hours. Memory leak suspected.", "priority": "Medium"},
    {"ticket": "Could you add keyboard shortcuts for common actions? Power users would appreciate this productivity feature.", "priority": "Medium"},
    {"ticket": "Drag and drop occasionally doesn't register first attempt. Need to try 2-3 times. Annoying but not blocking.", "priority": "Medium"},
    {"ticket": "Date picker defaults to US format. We're UK-based and would prefer DD/MM/YYYY. Localization request.", "priority": "Medium"},
    {"ticket": "Performance degrades with 1000+ items in a list. Pagination would help but not critical for most users.", "priority": "Medium"},
    
    # ----- Low：咨询 / 文档 / 外观 -----
    {"ticket": "Where can I find documentation for the API? Looking for endpoint reference. No urgency, just getting started.", "priority": "Low"},
    {"ticket": "Typo on the pricing page. 'Busines' instead of 'Business'. Minor cosmetic issue.", "priority": "Low"},
    {"ticket": "What's the difference between Pro and Enterprise plans? Looking to potentially upgrade in the future.", "priority": "Low"},
    {"ticket": "The favicon looks slightly blurry on retina displays. Very minor visual thing. Not affecting functionality.", "priority": "Low"},
    {"ticket": "Is there a way to change the accent color in the UI? Brand colors are blue but your default is purple.", "priority": "Low"},
    {"ticket": "How do I export my data if I want to cancel? Just asking for future reference, not canceling now.", "priority": "Low"},
    {"ticket": "Documentation example code is in Python but I use JavaScript. Request for JS examples would be helpful.", "priority": "Low"},
    {"ticket": "Logo in footer is the old version. We rebranded 6 months ago. Would be nice to update when convenient.", "priority": "Low"},
    {"ticket": "What browsers do you officially support? Just curious about your compatibility matrix.", "priority": "Low"},
    {"ticket": "The 'Learn More' link on the homepage goes to a page with not much info. Maybe add more content?", "priority": "Low"},
    {"ticket": "Font size in the footer is a bit small. Hard to read but not critical information anyway.", "priority": "Low"},
    {"ticket": "Do you have a referral program? Would like to recommend your service to colleagues if there's a benefit.", "priority": "Low"},
    {"ticket": "Inconsistent button styles on different pages. Some rounded, some square. Minor design inconsistency.", "priority": "Low"},
    {"ticket": "How long do you retain deleted data? Privacy policy wasn't clear on retention period.", "priority": "Low"},
    {"ticket": "Would be nice to have a changelog page showing recent updates. Just a suggestion for transparency.", "priority": "Low"},
    {"ticket": "Empty state message could be friendlier. 'No data' is accurate but cold. UX writing improvement.", "priority": "Low"},
    {"ticket": "Do you offer student discounts? I'm learning to use your platform for my thesis project.", "priority": "Low"},
    {"ticket": "The tutorial video is from 2022. UI has changed since then. Update when you have time.", "priority": "Low"},
]

# 打印总条数与各优先级分布，便于检查样本是否失衡
print(f"Total tickets in dataset: {len(TICKETS)}")
print(f"Priority distribution: {Counter(t['priority'] for t in TICKETS)}")


Total tickets in dataset: 60
Priority distribution: Counter({'Low': 18, 'Medium': 16, 'High': 14, 'Urgent': 12})


## 训练集 / 验证集 / 测试集划分

打乱后按约 **70% / 15% / 15%** 切开。本格用随机划分（固定 `seed=42` 可复现）；真正生产可再加分层抽样（stratify）以稳住各类占比。


In [4]:
# ========== 打乱并划分：70% train / 15% val / 15% test ==========

# 固定随机种子，保证每次划分一致，方便对比实验
random.seed(42)
# copy 后再 shuffle，避免打乱原始 TICKETS 顺序（若后面还要按原序展示）
shuffled = TICKETS.copy()
random.shuffle(shuffled)

# 总样本数
n = len(shuffled)
# 训练集长度
n_train = int(0.7 * n)
# 验证集长度（测试集 = 剩余）
n_val = int(0.15 * n)

# 切片：前 n_train → train；接着 n_val → val；其余 → test
train_data = shuffled[:n_train]
val_data = shuffled[n_train:n_train + n_val]
test_data = shuffled[n_train + n_val:]

# 打印各子集规模与标签分布
print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print(f"\nTrain distribution: {Counter(t['priority'] for t in train_data)}")
print(f"Test distribution: {Counter(t['priority'] for t in test_data)}")


Train: 42 | Val: 9 | Test: 9

Train distribution: Counter({'Low': 13, 'Medium': 12, 'High': 11, 'Urgent': 6})
Test distribution: Counter({'Urgent': 3, 'High': 3, 'Low': 2, 'Medium': 1})


## 零样本基线（Zero-Shot Baseline）

不微调、只靠 system prompt，让 frontier 模型直接分类。这个准确率是后面微调要「打过」的基准线。


In [5]:
# ========== 零样本分类：system 定规则，模型只回一个优先级词 ==========

# SYSTEM_PROMPT：发给模型的分流规则（英文保留——改译会改变分类行为）
SYSTEM_PROMPT = """You are a customer support ticket triage system.
Classify the support ticket into exactly one priority level:
- Urgent: System down, security breach, data loss, production outage
- High: Major feature broken, payment issues, significant user impact
- Medium: Feature request, minor bug, performance concern
- Low: General inquiry, documentation question, cosmetic issue

Respond with ONLY the priority level (Urgent, High, Medium, or Low), nothing else."""

def predict_baseline(ticket: str) -> str:
    """用 OpenRouter/OpenAI 做零样本分类，返回标准化优先级标签。"""
    try:
        # chat.completions.create：一次非流式补全
        response = client.chat.completions.create(
            model=FRONTIER_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": ticket}
            ],
            # 标签很短，限制 max_tokens 省钱也防胡说长文
            max_tokens=10,
            # temperature=0：尽量确定性，便于评估复现
            temperature=0
        )
        if response.choices:
            # 取出助手文本并去空白
            raw = (response.choices[0].message.content or "").strip()
            # 在回复里模糊匹配四个合法标签（大小写不敏感）
            for label in PRIORITIES:
                if label.lower() in raw.lower():
                    return label
            # 匹配不到就原样返回；空串则默认 Medium
            return raw or "Medium"
        return "Medium"
    except Exception as e:
        # API 失败时打印错误并回退 Medium，避免整次评估崩掉
        print(f"Prediction error: {e}")
        return "Medium"

# 用测试集第一条做冒烟测试
test_ticket = test_data[0]
prediction = predict_baseline(test_ticket["ticket"])
print(f"Sample ticket: {test_ticket['ticket'][:100]}...")
print(f"True: {test_ticket['priority']} | Predicted: {prediction}")


Prediction error: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "Invalid \'max_output_tokens\': integer below minimum value. Expected a value >= 16, but got 10 instead.",\n    "type": "invalid_request_error",\n    "param": "max_output_tokens",\n    "code": "integer_below_min_value"\n  }\n}', 'provider_name': 'Azure', 'is_byok': False}}, 'user_id': 'user_39l2mYQ8CncrKVDueHpsmmP8E63'}
Sample ticket: Would be nice to have a changelog page showing recent updates. Just a suggestion for transparency....
True: Low | Predicted: Medium


In [6]:
# ========== 评估准确率：逐条预测并汇总 ==========

def calculate_accuracy(predictor, data, verbose=False):
    """在 data 上跑 predictor，返回 (accuracy, 明细列表)。"""
    correct = 0
    results = []
    
    for item in data:
        # 对工单正文做预测
        pred = predictor(item["ticket"])
        # 与真值标签比是否完全一致
        is_correct = pred == item["priority"]
        if is_correct:
            correct += 1
        # 明细：截断正文便于打印
        results.append({
            "ticket": item["ticket"][:80] + "...",
            "true": item["priority"],
            "predicted": pred,
            "correct": is_correct
        })
        if verbose:
            # OK / WRONG 状态行（字符串保留英文，依赖程序判断可改但不必要）
            status = "OK" if is_correct else "WRONG"
            print(f"[{status}] True: {item['priority']:6} | Pred: {pred:6} | {item['ticket'][:60]}...")
    
    # 空集保护：避免除零
    accuracy = correct / len(data) if data else 0.0
    return accuracy, results

# 在测试集上评估基线（verbose 打印每一条）
print("Evaluating baseline model on test set...\n")
baseline_accuracy, baseline_results = calculate_accuracy(predict_baseline, test_data, verbose=True)
print(f"\n=== Baseline Accuracy: {baseline_accuracy:.1%} ===")
print("This is the number to beat with fine-tuning!")


Evaluating baseline model on test set...

Prediction error: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "Invalid \'max_output_tokens\': integer below minimum value. Expected a value >= 16, but got 10 instead.",\n    "type": "invalid_request_error",\n    "param": "max_output_tokens",\n    "code": "integer_below_min_value"\n  }\n}', 'provider_name': 'Azure', 'is_byok': False}}, 'user_id': 'user_39l2mYQ8CncrKVDueHpsmmP8E63'}
[WRONG] True: Low    | Pred: Medium | Would be nice to have a changelog page showing recent update...
Prediction error: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "Invalid \'max_output_tokens\': integer below minimum value. Expected a value >= 16, but got 10 instead.",\n    "type": "invalid_request_error",\n    "param": "max_output_tokens",\n    "code": "integer_below_min_value"\n  }\n}', 'pr

## 准备微调数据（JSONL）

把 train / val 导出成 **JSONL**（每行一个 JSON）：`messages` 里是 system + user(工单) + assistant(优先级)。这是 OpenAI 等微调 API 的常见格式。


In [7]:
# ========== 构造微调 messages，并预览一条 JSONL ==========

def messages_for(item):
    """一条训练样本：user=工单正文，assistant=优先级标签。"""
    return [
        {"role": "system", "content": "Classify the support ticket priority. Respond with only: Urgent, High, Medium, or Low."},
        {"role": "user", "content": item["ticket"]},
        {"role": "assistant", "content": item["priority"]}
    ]

def make_jsonl(items):
    """把多条样本变成 JSONL 字符串（行与行之间用换行分隔）。"""
    lines = [json.dumps({"messages": messages_for(item)}) for item in items]
    return "\n".join(lines)

def write_jsonl(items, filepath):
    """写入 JSONL 文件；若目录不存在则创建。"""
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(make_jsonl(items))

# 先打印训练集第一条，确认格式长什么样
print("Example JSONL entry:")
print(json.dumps({"messages": messages_for(train_data[0])}, indent=2))


Example JSONL entry:
{
  "messages": [
    {
      "role": "system",
      "content": "Classify the support ticket priority. Respond with only: Urgent, High, Medium, or Low."
    },
    {
      "role": "user",
      "content": "Could you add keyboard shortcuts for common actions? Power users would appreciate this productivity feature."
    },
    {
      "role": "assistant",
      "content": "Medium"
    }
  ]
}


In [ ]:
# ========== 导出 train / validation 两个 JSONL 文件 ==========

# 输出目录名
JSONL_DIR = "jsonl"
# 确保目录存在
os.makedirs(JSONL_DIR, exist_ok=True)

# 写训练集
write_jsonl(train_data, f"{JSONL_DIR}/fine_tune_train.jsonl")
# 写验证集（微调时可做 early stopping / 监控）
write_jsonl(val_data, f"{JSONL_DIR}/fine_tune_validation.jsonl")

# 提示文件位置与可用工具（print 文案保留英文）
print(f"Exported training data to {JSONL_DIR}/")
print(f"  - fine_tune_train.jsonl: {len(train_data)} examples")
print(f"  - fine_tune_validation.jsonl: {len(val_data)} examples")
print(f"\nThese files can be used with:")
print(f"  - OpenAI fine-tuning API (platform.openai.com)")
print(f"  - Open-source fine-tuning tools (Hugging Face, Axolotl, etc.)")


## （可选）用 LLM 再生成更多训练工单

数据少时，可让同一个 frontier 模型按优先级定义批量「造」工单，再并进 `TICKETS` 重新划分。


In [8]:
# ========== （可选）让模型按四档定义批量生成合成工单 ==========

def generate_tickets(n_per_priority: int = 3):
    """调用 LLM 为每个优先级生成 n_per_priority 条工单，返回合法 dict 列表。"""
    # prompt 整段英文保留：规定 JSON 数组输出格式
    prompt = f"""Generate {n_per_priority} realistic customer support tickets for each priority level.

Priority definitions:
- Urgent: System down, security breach, data loss, production outage
- High: Major feature broken, payment issues, significant user impact  
- Medium: Feature request, minor bug, performance concern
- Low: General inquiry, documentation question, cosmetic issue

Reply with ONLY a JSON array, no other text:
[{{"ticket": "ticket text here", "priority": "Urgent"}}, ...]"""
    
    try:
        response = client.chat.completions.create(
            model=FRONTIER_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=2000,
            temperature=0.7
        )
        if response.choices:
            raw = (response.choices[0].message.content or "").strip()
            # 去掉模型可能包上的 ```json ... ``` 围栏
            raw = re.sub(r"^```(?:json)?\s*", "", raw).strip()
            raw = re.sub(r"\s*```$", "", raw).strip()
            
            # 解析 JSON 数组
            generated = json.loads(raw)
            # 只保留结构合法且 priority 在 PRIORITIES 里的条目
            valid = [
                g for g in generated 
                if isinstance(g, dict) and g.get("priority") in PRIORITIES and g.get("ticket")
            ]
            return valid
    except Exception as e:
        print(f"Generation failed: {e}")
    return []

# 需要更多数据时取消下面三行注释：
# new_tickets = generate_tickets(3)
# print(f"Generated {len(new_tickets)} new tickets")
# TICKETS.extend(new_tickets)


## Gradio UI：交互式工单分类器

界面提供：

- 单条工单分类
- 测试集批量评估
- 预测与真值对照


In [9]:
# ========== Gradio 回调：单条分类 / 批量评估 / 样例 / 导出状态 ==========

# 优先级 → 展示用颜色（CSS hex）
PRIORITY_COLORS = {
    "Urgent": "#dc3545",  # 红
    "High": "#fd7e14",    # 橙
    "Medium": "#ffc107",  # 黄
    "Low": "#28a745"      # 绿
}

def classify_ticket(ticket_text: str) -> str:
    """单条分类：调用基线模型，返回带颜色的 Markdown 结果。"""
    if not ticket_text.strip():
        return "Please enter a ticket description."
    
    priority = predict_baseline(ticket_text)
    color = PRIORITY_COLORS.get(priority, "#6c757d")
    
    # 返回给 Gradio Markdown 组件的 HTML/MD（英文 UI 文案保留）
    return f"""## Classification Result

**Priority:** <span style="color:{color}; font-weight:bold; font-size:1.2em;">{priority}</span>

### Priority Definitions:
- **Urgent**: System down, security breach, data loss
- **High**: Major feature broken, payment issues
- **Medium**: Feature request, minor bug
- **Low**: General inquiry, documentation question"""

def run_evaluation() -> str:
    """在测试集上跑基线，拼出准确率表与分优先级统计。"""
    accuracy, results = calculate_accuracy(predict_baseline, test_data)
    
    # 表头 + 总体指标
    output = f"""## Evaluation Results

**Model:** {FRONTIER_MODEL}  
**Test Set Size:** {len(test_data)}  
**Accuracy:** {accuracy:.1%}

### Sample Predictions:

| Status | True | Predicted | Ticket |
|--------|------|-----------|--------|
"""
    
    # 只展示前 10 条明细，避免 UI 过长
    for r in results[:10]:
        status = "OK" if r["correct"] else "WRONG"
        output += f"| {status} | {r['true']} | {r['predicted']} | {r['ticket'][:50]}... |\n"
    
    # 按真值优先级汇总正确率
    by_priority = {}
    for r in results:
        key = r["true"]
        if key not in by_priority:
            by_priority[key] = {"correct": 0, "total": 0}
        by_priority[key]["total"] += 1
        if r["correct"]:
            by_priority[key]["correct"] += 1
    
    output += "\n### Accuracy by Priority:\n\n"
    for priority in PRIORITIES:
        if priority in by_priority:
            stats = by_priority[priority]
            pct = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
            output += f"- **{priority}**: {stats['correct']}/{stats['total']} ({pct:.0f}%)\n"
    
    return output

def get_sample_ticket(priority: str) -> str:
    """从数据集里随机抽一条指定优先级的工单，填进输入框。"""
    matching = [t for t in TICKETS if t["priority"] == priority]
    if matching:
        return random.choice(matching)["ticket"]
    return ""

def export_data() -> str:
    """检查 JSONL 是否已导出，并提示微调下一步。"""
    train_path = f"{JSONL_DIR}/fine_tune_train.jsonl"
    val_path = f"{JSONL_DIR}/fine_tune_validation.jsonl"
    
    train_exists = os.path.exists(train_path)
    val_exists = os.path.exists(val_path)
    
    return f"""## JSONL Export Status

| File | Status | Examples |
|------|--------|----------|
| {train_path} | {'Exists' if train_exists else 'Missing'} | {len(train_data)} |
| {val_path} | {'Exists' if val_exists else 'Missing'} | {len(val_data)} |

### Next Steps for Fine-Tuning:

1. **OpenAI Fine-Tuning:**
   ```python
   openai.files.create(file=open("{train_path}", "rb"), purpose="fine-tune")
   openai.fine_tuning.jobs.create(training_file=file_id, model="gpt-4.1-nano-2025-04-14")
   ```

2. **Open-Source Fine-Tuning:**
   - Use with Hugging Face Transformers
   - Compatible with Axolotl, LLaMA-Factory, etc.
"""


In [10]:
# ========== 组装 Gradio Blocks：分类 / 评估 / 导出 / 数据集信息 ==========

# Soft 主题的多页签界面
with gr.Blocks(title="Support Ticket Classifier", theme=gr.themes.Soft()) as demo:
    # 顶栏说明（UI 英文保留）
    gr.Markdown("""# Customer Support Ticket Priority Classifier
    
Classify support tickets into priority levels using a frontier LLM.
This demonstrates the baseline performance before fine-tuning.""")
    
    with gr.Tabs():
        # ----- 页签 1：单条分类 -----
        with gr.Tab("Classify Ticket"):
            with gr.Row():
                with gr.Column():
                    ticket_input = gr.Textbox(
                        label="Ticket Description",
                        placeholder="Enter the support ticket text here...",
                        lines=5
                    )
                    with gr.Row():
                        classify_btn = gr.Button("Classify", variant="primary")
                        clear_btn = gr.Button("Clear")
                    
                    gr.Markdown("### Load Sample Ticket:")
                    with gr.Row():
                        # 四个优先级按钮：点击填入对应样例工单
                        for priority in PRIORITIES:
                            btn = gr.Button(priority, size="sm")
                            btn.click(
                                fn=lambda p=priority: get_sample_ticket(p),
                                outputs=ticket_input
                            )
                
                with gr.Column():
                    result_output = gr.Markdown(label="Result")
            
            # 分类 / 清空
            classify_btn.click(fn=classify_ticket, inputs=ticket_input, outputs=result_output)
            clear_btn.click(fn=lambda: ("", ""), outputs=[ticket_input, result_output])
        
        # ----- 页签 2：批量评估基线 -----
        with gr.Tab("Evaluate Model"):
            gr.Markdown("""### Baseline Evaluation
            
Run the zero-shot classifier on the test set to measure baseline accuracy.
This is the number to beat with fine-tuning!""")
            
            eval_btn = gr.Button("Run Evaluation", variant="primary")
            eval_output = gr.Markdown()
            
            eval_btn.click(fn=run_evaluation, outputs=eval_output)
        
        # ----- 页签 3：查看 JSONL 导出状态 -----
        with gr.Tab("Export JSONL"):
            gr.Markdown("""### Training Data Export
            
View the exported JSONL files for fine-tuning.""")
            
            export_btn = gr.Button("Check Export Status", variant="primary")
            export_output = gr.Markdown()
            
            export_btn.click(fn=export_data, outputs=export_output)
        
        # ----- 页签 4：数据集与模型配置一览 -----
        with gr.Tab("Dataset Info"):
            dataset_info = f"""### Dataset Statistics

| Split | Count |
|-------|-------|
| Total | {len(TICKETS)} |
| Train | {len(train_data)} |
| Validation | {len(val_data)} |
| Test | {len(test_data)} |

### Priority Distribution (Full Dataset):

| Priority | Count | Percentage |
|----------|-------|------------|
"""
            counts = Counter(t['priority'] for t in TICKETS)
            for priority in PRIORITIES:
                count = counts.get(priority, 0)
                pct = count / len(TICKETS) * 100
                dataset_info += f"| {priority} | {count} | {pct:.1f}% |\n"
            
            dataset_info += f"""\n### Model Configuration

- **Model:** {FRONTIER_MODEL}
- **API:** {'OpenRouter' if 'openrouter' in OPENROUTER_BASE_URL else 'OpenAI'}
- **Temperature:** 0 (deterministic)
"""
            gr.Markdown(dataset_info)

# 启动 Gradio（本地会打印 URL）
demo.launch()


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Prediction error: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "Invalid \'max_output_tokens\': integer below minimum value. Expected a value >= 16, but got 10 instead.",\n    "type": "invalid_request_error",\n    "param": "max_output_tokens",\n    "code": "integer_below_min_value"\n  }\n}', 'provider_name': 'Azure', 'is_byok': False}}, 'user_id': 'user_39l2mYQ8CncrKVDueHpsmmP8E63'}
Prediction error: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "Invalid \'max_output_tokens\': integer below minimum value. Expected a value >= 16, but got 10 instead.",\n    "type": "invalid_request_error",\n    "param": "max_output_tokens",\n    "code": "integer_below_min_value"\n  }\n}', 'provider_name': 'Azure', 'is_byok': False}}, 'user_id': 'user_39l2mYQ8CncrKVDueHpsmmP8E63'}
Prediction error: Error code: 400 - {'error': {'messag

## （可选）用 OpenAI 正式微调

若已有 `OPENAI_API_KEY`，可对导出的 JSONL 发起 fine-tuning job（下一格代码默认注释在三引号里，需时再打开）。


In [ ]:
# ========== （可选）OpenAI 微调：上传 JSONL → 创建 fine-tuning job ==========
# 下面整段包在三引号里：默认不执行；需要时删掉三引号并确保 OPENAI_API_KEY 可用
# Requires: OPENAI_API_KEY environment variable

'''
from openai import OpenAI

# 初始化 OpenAI client (not OpenRouter)
openai_client = OpenAI()  # Uses OPENAI_API_KEY

# Upload training file
with open(f"{JSONL_DIR}/fine_tune_train.jsonl", "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Training file uploaded: {train_file.id}")

# Upload validation file
with open(f"{JSONL_DIR}/fine_tune_validation.jsonl", "rb") as f:
    val_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Validation file uploaded: {val_file.id}")

# Create fine-tuning job
job = openai_client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    hyperparameters={"n_epochs": 3},
    suffix="ticket-classifier"
)
print(f"Fine-tuning job created: {job.id}")

# Check job status
# openai_client.fine_tuning.jobs.retrieve(job.id)
'''


## 小结

本练习串起微调前的完整准备：

1. **数据准备**：合成带标签的客服工单
2. **Train/Val/Test**：可复现划分，便于公平评估
3. **零样本基线**：不微调时 frontier 模型的准确率（要打过的数）
4. **JSONL 导出**：对接 OpenAI / 开源微调工具
5. **Gradio UI**：单条分类 + 批量评估

### 关键指标

- **Baseline Accuracy**：零样本分类准确率
- **目标**：微调后应高于基线

### 下一步

1. 用 OpenAI 或开源工具真正跑微调
2. 对比微调模型 vs 基线
3. 可尝试：更多数据、不同基座、调 epochs / batch size
